# Individual Composition Analysis of the SymbTr v3.0 TXT Dataset

## Overview

This notebook performs an independent statistical analysis of every symbolic music composition contained in the **SymbTr v3.0 TXT dataset**.

Each TXT file represents a single musical composition and is processed separately using an identical analytical workflow. Rather than merging all compositions into a single dataset, every piece is analyzed individually to preserve its musical characteristics and structural properties.

For each composition, the notebook extracts descriptive metadata from the filename, reads the symbolic note sequence, computes statistical characteristics of pitch, frequency, and rhythmic duration, and produces a standardized analytical profile. Applying the same methodology to every composition ensures consistency across the entire dataset and enables reliable comparative analyses in subsequent notebooks.

The outputs generated in this notebook constitute the foundation for corpus-level statistical summaries, visualization, clustering, similarity analysis, and machine learning applications presented later in this analysis book.

## Importing the Required Libraries

This notebook relies on several Python libraries to support data acquisition, file management, numerical computation, statistical analysis, and visualization.

- **Pathlib** provides platform-independent file and directory management.
- **Requests** downloads the SymbTr dataset from Zenodo.
- **ZipFile** extracts the downloaded archive.
- **Pandas** manages tabular data structures.
- **NumPy** performs numerical computations.
- **Collections** supports frequency-based analyses.
- **Matplotlib** is used to generate publication-quality visualizations.

These libraries establish the computational environment required for processing every symbolic music composition contained in the dataset.

In [19]:
from pathlib import Path
from collections import Counter
import zipfile

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Environment initialized successfully.")

Environment initialized successfully.


## Initializing the Project Data Directories

This notebook uses the standardized directory structure adopted throughout the **TDC Analysis Book**.

The raw data directory stores the original SymbTr dataset downloaded from Zenodo, while the interim and processed directories are used to store temporary files and analysis outputs generated during the workflow.

Creating these directories automatically ensures that the notebook can be executed repeatedly without requiring manual file management and provides a consistent environment for subsequent analyses.

In [20]:
from pathlib import Path

# Zenodo record identifier
record_id = "15470412"

# Zenodo REST API endpoint
api_url = f"https://zenodo.org/api/records/{record_id}"

# Project data directories
project_root = Path.cwd().resolve().parents[1]

data_directory = project_root / "data"

raw_data_directory = data_directory / "raw"
interim_data_directory = data_directory / "interim"
processed_data_directory = data_directory / "processed"

# SymbTr TXT dataset paths
txt_archive_path = raw_data_directory / "txt_v3.zip"
txt_data_directory = raw_data_directory / "txt_v3"

# Create project directories if they do not already exist
for directory in (
    raw_data_directory,
    interim_data_directory,
    processed_data_directory,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project data directories initialized successfully.")

Project data directories initialized successfully.


## Downloading the SymbTr v3.0 TXT Dataset

The SymbTr symbolic music dataset is distributed through the Zenodo research data repository.

Before downloading, the notebook verifies whether the archive already exists in the project's raw data directory. If the archive is available locally, the download step is skipped to avoid unnecessary network traffic and to improve reproducibility.

The dataset is downloaded using streamed data transfer, allowing large files to be saved efficiently without loading the entire archive into memory.

In [21]:
# Official download address of the SymbTr TXT dataset
txt_download_url = (
    "https://zenodo.org/records/15470412/files/"
    "txt_v3.zip?download=1"
)

if txt_archive_path.exists():

    print("SymbTr TXT archive already exists.")

else:

    print("Downloading SymbTr TXT dataset...")

    response = requests.get(
        txt_download_url,
        stream=True,
        timeout=120,
    )

    response.raise_for_status()

    with open(txt_archive_path, "wb") as file:

        for chunk in response.iter_content(chunk_size=8192):

            if chunk:

                file.write(chunk)

    print("Download completed successfully.")

SymbTr TXT archive already exists.


## Extracting the Dataset

The downloaded ZIP archive is extracted into the project's raw data directory.

Before extraction, the notebook checks whether the symbolic music files have already been extracted. If the extraction has previously been completed, the operation is skipped automatically.

Each extracted TXT file corresponds to one symbolic representation of a Turkish makam composition.

In [22]:
existing_files = list(txt_data_directory.rglob("*.txt"))

if len(existing_files) == 0:

    print("Extracting dataset...")

    with zipfile.ZipFile(txt_archive_path, "r") as zip_ref:

        zip_ref.extractall(txt_data_directory)

    print("Extraction completed successfully.")

else:

    print("Dataset has already been extracted.")

Dataset has already been extracted.


## Identifying the Symbolic Music Files

After extraction, the notebook searches the dataset directory recursively and identifies every TXT file contained in the SymbTr collection.

Each TXT file represents one musical composition encoded using the SymbTr symbolic notation format. The complete collection of identified files will subsequently be processed independently, ensuring that identical analyses are performed for every composition in the dataset.

In [23]:
txt_files = sorted(txt_data_directory.rglob("*.txt"))

print(f"Total symbolic music compositions: {len(txt_files)}")

Total symbolic music compositions: 3000


## Inspecting the Dataset

Before beginning the analyses, it is useful to verify that all symbolic music files have been located successfully.

The following cell reports the total number of compositions available in the dataset and displays several example filenames. Since each filename encodes musical metadata, this preliminary inspection also provides an overview of the naming convention adopted by the SymbTr corpus.

In [24]:
print(f"Number of compositions : {len(txt_files)}")

print("\nExample compositions:\n")

for file in txt_files[:10]:
    print(file.name)

Number of compositions : 3000

Example compositions:

acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
acem--ilahi--nimevsat--calabim_bir--haci_bayram_veli.txt
acem--kupe--duyek--zulfunu--ahmet_avni_konuk.txt
acem--selam--devrikebir--asik-i_ger--huseyin_fahreddin_dede.txt
acem--seyir--sofyan----sefik_gurmeric.txt
acem--seyir--sofyan--1--erol_bingol.txt
acem--turku--duyek--ordunun_dereleri--ordu.txt
acemasiran--agirsemai--senginsemai--ey_lebleri--dede_efendi.txt
acemasiran--aranagme--agiraksak--1--.txt
acemasiran--aranagme--aksak--1--.txt


## Understanding the Filename Structure

The SymbTr dataset adopts a structured filename convention in which descriptive musical metadata are encoded directly within each filename.

These metadata include:

- Makam
- Musical form
- Usul (rhythmic cycle)
- Composition title
- Composer

The filename metadata are extracted automatically and combined with the statistical information computed from the corresponding TXT file. While the filename provides descriptive information about the composition, the musical analysis itself is performed using the symbolic note data stored in the TXT file (e.g., Nota53, Koma53, Pay, Payda, and related fields).

## Extracting Metadata from the Filename

Each symbolic music composition in the SymbTr dataset is associated with a structured filename that encodes important descriptive metadata about the corresponding musical work. Instead of treating filenames solely as identifiers, the metadata extraction process interprets each filename as a source of musical information.

The filename is parsed by removing the file extension and separating its predefined components according to the filename convention used in the dataset. Each component is then assigned to its corresponding metadata field, including the makam (melodic mode), musical form, rhythmic cycle (usul), composition title, and composer.

For example, consider the following **illustrative** filename:

```text
hicaz--sarki--aksak--bir_ornek_eser--bestekar.txt
```

After parsing the filename, the extracted metadata can be represented as:

| Filename Component | Metadata Field |
|--------------------|----------------|
| `hicaz` | Makam |
| `sarki` | Musical form |
| `aksak` | Usul (rhythmic cycle) |
| `bir_ornek_eser` | Composition title |
| `bestekar` | Composer |

The extracted metadata are stored in a structured Python dictionary and later combined with the statistical descriptors computed from the symbolic note information contained in the corresponding TXT file. While the filename provides descriptive information about the composition, the musical analysis itself is performed using the symbolic data stored in the TXT file, including fields such as **Nota53**, **Koma53**, **Pay**, **Payda**, **Ms**, and other symbolic attributes.

Combining descriptive metadata with statistical features enables each composition to be analyzed not only according to its symbolic musical content but also within its broader musical context, facilitating subsequent statistical analyses, visualization, clustering, and machine learning applications.

### Extracting Composition Metadata

Each SymbTr TXT file has a structured filename that contains descriptive metadata about the corresponding musical composition. The code below defines the `extract_metadata()` function, which removes the file extension, splits the filename into its predefined components using the `"--"` delimiter, maps each component to its corresponding metadata field (makam, musical form, usul, title, and composer), and stores the extracted information in a Python dictionary.

The resulting dictionary provides a structured representation of the composition metadata, which can be easily integrated with the statistical features computed from the symbolic note data contained in the corresponding TXT file. Combining descriptive metadata with symbolic musical features enables each composition to be analyzed within its musical context and supports subsequent statistical analysis, visualization, clustering, and machine learning applications.

In [25]:
def extract_metadata(file_path):
    """
    Extract metadata encoded in a SymbTr filename.
    """

    parts = file_path.stem.split("--")

    metadata = {
        "makam": parts[0] if len(parts) > 0 else None,
        "form": parts[1] if len(parts) > 1 else None,
        "usul": parts[2] if len(parts) > 2 else None,
        "title": parts[3] if len(parts) > 3 else None,
        "composer": parts[4] if len(parts) > 4 else None,
    }

    return metadata


For example, the illustrative filename

```text
hicaz--sarki--aksak--bir_ornek_eser--bestekar.txt
```

is converted into the following metadata dictionary:

```python
{
    "makam": "hicaz",
    "form": "sarki",
    "usul": "aksak",
    "title": "bir_ornek_eser",
    "composer": "bestekar"
}
```

## Reading and Preprocessing Symbolic Music Files

Each SymbTr TXT file contains the symbolic representation of a single Turkish makam composition, including note names, pitch values represented in the **Koma53** system, and symbolic duration information.

The function below reads an individual TXT file and prepares it for subsequent statistical analysis. The preprocessing workflow consists of the following steps:

- reads the symbolic music file using several alternative text encodings to ensure compatibility with different file formats;
- standardizes column names by removing unnecessary whitespace;
- verifies that the required variables (`NotaAE`, `Koma53`, `Pay`, and `Payda`) are present;
- retains only the variables required for the analysis;
- renames selected variables to more descriptive names (`Note`, `Numerator`, and `Denominator`);
- converts numerical variables to appropriate numeric data types;
- cleans note labels by removing unnecessary whitespace;
- removes incomplete observations and invalid records, including notes with missing values or zero denominators;
- computes the **UnitDuration** variable as the ratio of **Numerator** to **Denominator**, providing a normalized symbolic duration for every note.

The function returns a cleaned and standardized Pandas DataFrame representing a single musical composition. This preprocessing step ensures that every SymbTr composition follows a consistent structure before the extraction of statistical features in the subsequent analysis.

| Step | Description |
|------|-------------|
| **1. Read the TXT file** | Attempts to read the SymbTr TXT file using multiple character encodings to ensure compatibility with differently encoded files. |
| **2. Standardize column names** | Removes unnecessary whitespace from the column names to ensure consistent variable matching. |
| **3. Validate required columns** | Checks whether the required variables (`NotaAE`, `Koma53`, `Pay`, and `Payda`) are present before continuing the preprocessing procedure. |
| **4. Select analysis variables** | Retains only the variables required for the statistical analysis and excludes the remaining columns. |
| **5. Rename variables** | Renames `NotaAE`, `Pay`, and `Payda` as `Note`, `Numerator`, and `Denominator` to improve readability. |
| **6. Convert data types** | Converts `Koma53`, `Numerator`, and `Denominator` to numeric data types. Invalid values are converted to missing values. |
| **7. Clean note labels** | Converts note labels to a consistent string format and removes unnecessary leading or trailing whitespace. |
| **8. Remove invalid observations** | Removes rows with missing pitch or duration values and excludes observations with a zero denominator. |
| **9. Compute unit durations** | Calculates the symbolic duration of each note as the ratio of the duration numerator (`Pay`) to the duration denominator (`Payda`). |
| **10. Return the processed data** | Returns a clean and standardized Pandas DataFrame representing a single symbolic music composition, ready for statistical analysis. |

In [26]:
def read_symbtr_txt(file_path):
    """
    Read and preprocess a single SymbTr TXT composition.

    Parameters
    ----------
    file_path : pathlib.Path
        Path to the SymbTr TXT file.

    Returns
    -------
    pandas.DataFrame
        Cleaned symbolic music data containing note labels, Koma53 pitch
        values, duration components, and computed unit durations.
    """

    # 1. Read the TXT file using alternative encodings
    encodings = [
        "utf-8-sig",
        "cp1254",
        "cp1252",
        "latin-1",
    ]

    df = None
    last_error = None

    for encoding in encodings:
        try:
            df = pd.read_csv(
                file_path,
                sep="\t",
                header=0,
                encoding=encoding,
                engine="python",
                quoting=3,
                on_bad_lines="skip",
            )
            break

        except (UnicodeDecodeError, pd.errors.ParserError) as error:
            last_error = error

    if df is None:
        raise ValueError(
            f"File could not be read: {file_path.name}. "
            f"Last error: {last_error}"
        )

    # 2. Standardize column names
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    # 3. Validate required columns
    required_columns = [
        "NotaAE",
        "Koma53",
        "Pay",
        "Payda",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns in {file_path.name}: "
            f"{missing_columns}"
        )

    # 4. Select variables required for the analysis
    df = df[
        [
            "NotaAE",
            "Koma53",
            "Pay",
            "Payda",
        ]
    ].copy()

    # 5. Rename selected variables
    df = df.rename(
        columns={
            "NotaAE": "Note",
            "Pay": "Numerator",
            "Payda": "Denominator",
        }
    )

    # 6. Convert numerical variables to numeric data types
    numeric_columns = [
        "Koma53",
        "Numerator",
        "Denominator",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    # 7. Clean note labels
    df["Note"] = (
        df["Note"]
        .astype("string")
        .str.strip()
    )

    # 8. Remove incomplete and invalid observations
    df = df.dropna(
        subset=[
            "Koma53",
            "Numerator",
            "Denominator",
        ]
    )

    df = df[
        df["Denominator"] != 0
    ].copy()

    # 9. Compute symbolic unit durations
    df["UnitDuration"] = (
        df["Numerator"]
        / df["Denominator"]
    )

    # 10. Return the processed composition data
    return df

## Computing Statistical Profiles of Musical Compositions

Each SymbTr TXT file represents a single Turkish makam composition. While the symbolic note sequence provides detailed musical information, statistical descriptors offer a compact summary of the composition that can be compared across the entire corpus.

The function below computes an independent statistical profile for each composition by combining the extracted filename metadata with descriptive statistics calculated from the symbolic music data. These statistics characterize different musical properties, including the overall size of the composition, the distribution of symbolic pitch values, rhythmic duration patterns, and note usage.

The computed variables provide a standardized numerical representation of each composition. Such representations are useful for descriptive corpus analysis, visualization, similarity analysis, clustering, dimensionality reduction, and machine learning applications, where each musical piece must be represented by a consistent set of quantitative features.

The statistical profile consists of four groups of variables:

- **Composition metadata**, including the makam, musical form, usul, title, composer, and filename.
- **General note statistics**, describing the number of notes and the diversity of note symbols used in the composition.
- **Symbolic pitch statistics**, summarizing the distribution of the **Koma53** pitch values through measures such as minimum, maximum, mean, median, standard deviation, and range.
- **Symbolic duration statistics**, describing rhythmic characteristics using the symbolic unit durations derived from the **Pay** and **Payda** values.
- **Note occurrence statistics**, identifying the most frequently occurring symbolic note and its frequency within the composition.

Together, these statistical descriptors transform each symbolic music file into a structured feature vector, enabling quantitative comparison and large-scale computational analysis of Turkish makam music.

In [27]:
def analyze_composition(file_path):
    """
    Compute an independent statistical profile for one SymbTr composition.

    Parameters
    ----------
    file_path : pathlib.Path
        Path to a SymbTr TXT composition file.

    Returns
    -------
    dict
        Metadata and descriptive statistics computed from the symbolic
        pitch, note, and duration information of the composition.
    """

    metadata = extract_metadata(file_path)
    notes = read_symbtr_txt(file_path)

    # Remove missing or empty note labels
    valid_notes = (
        notes["Note"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    valid_notes = valid_notes[
        valid_notes != ""
    ]

    # Determine the most frequently occurring note
    note_counts = valid_notes.value_counts()

    if note_counts.empty:
        most_common_note = None
        most_common_note_count = 0
    else:
        most_common_note = note_counts.index[0]
        most_common_note_count = int(
            note_counts.iloc[0]
        )

    # Compute the statistical profile
    statistics = {
        **metadata,

        "filename": file_path.name,

        # General note statistics
        "note_count": int(len(notes)),
        "unique_note_count": int(
            valid_notes.nunique()
        ),

        # Symbolic pitch statistics
        "minimum_koma53": notes["Koma53"].min(),
        "maximum_koma53": notes["Koma53"].max(),
        "mean_koma53": notes["Koma53"].mean(),
        "median_koma53": notes["Koma53"].median(),
        "std_koma53": notes["Koma53"].std(),
        "koma53_range": (
            notes["Koma53"].max()
            - notes["Koma53"].min()
        ),

        # Symbolic duration statistics
        "total_unit_duration": (
            notes["UnitDuration"].sum()
        ),
        "minimum_unit_duration": (
            notes["UnitDuration"].min()
        ),
        "maximum_unit_duration": (
            notes["UnitDuration"].max()
        ),
        "mean_unit_duration": (
            notes["UnitDuration"].mean()
        ),
        "median_unit_duration": (
            notes["UnitDuration"].median()
        ),
        "std_unit_duration": (
            notes["UnitDuration"].std()
        ),

        # Note occurrence statistics
        "most_common_note": most_common_note,
        "most_common_note_count": (
            most_common_note_count
        ),
    }

    return statistics

## Analyzing Every Composition Independently

The validated analysis pipeline is now applied independently to every SymbTr TXT file contained in the dataset.

For each composition, the workflow performs the following operations:

- reads the symbolic music file;
- extracts the metadata encoded in the filename;
- computes descriptive statistics from the symbolic pitch (**Koma53**), note, and duration information;
- generates a statistical profile representing the individual musical composition;
- saves the resulting profile as an independent CSV file named **`<composition_name>_analysis.csv`** (e.g., `rast_sarki_analysis.csv`).

During processing, the notebook continuously reports the analysis progress by indicating how many compositions have been processed. The entire workflow is enclosed within an exception-handling mechanism to ensure that unreadable or irregular files do not interrupt the analysis of the remaining compositions. Files that cannot be processed are recorded separately together with the corresponding error information, enabling subsequent inspection and quality control.

The extracted statistical profiles are stored both in memory and as individual CSV files. Each analysis file is saved in the **`data/processed/individual_composition_analyses`** directory, preserving a direct correspondence between each symbolic music file and its statistical representation. This organization improves traceability, facilitates reproducibility, and provides standardized feature representations that can subsequently be integrated for corpus-level statistical analysis, visualization, clustering, and machine learning applications.

**Output.** This cell analyzes every SymbTr TXT file independently, generates a statistical profile for each composition, and saves the results as individual CSV files named **`<composition_name>_analysis.csv`** in the **`data/processed/individual_composition_analyses`** directory. Any files that cannot be processed are recorded together with the corresponding error information.

## Variables Included in Each Composition Analysis File

Each generated **`<composition_name>_analysis.csv`** file contains a statistical summary describing a single SymbTr composition. The extracted variables are listed below.

| Variable | Description |
|----------|-------------|
| `makam` | Makam identified from the filename. |
| `form` | Musical form identified from the filename. |
| `usul` | Rhythmic cycle (usul) identified from the filename. |
| `title` | Composition title extracted from the filename. |
| `composer` | Composer name extracted from the filename. |
| `filename` | Original SymbTr TXT filename used for the analysis. |
| `note_count` | Total number of symbolic notes contained in the composition. |
| `unique_note_count` | Number of distinct symbolic note names appearing in the composition. |
| `minimum_koma53` | Lowest pitch value represented in the 53-comma pitch system (Koma53). |
| `maximum_koma53` | Highest pitch value represented in the 53-comma pitch system (Koma53). |
| `mean_koma53` | Mean Koma53 pitch value across all notes. |
| `median_koma53` | Median Koma53 pitch value. |
| `std_koma53` | Standard deviation of the Koma53 pitch values, indicating pitch dispersion. |
| `koma53_range` | Difference between the highest and lowest Koma53 pitch values. |
| `total_unit_duration` | Sum of the symbolic note durations in the composition. |
| `minimum_unit_duration` | Shortest symbolic note duration. |
| `maximum_unit_duration` | Longest symbolic note duration. |
| `mean_unit_duration` | Mean symbolic note duration. |
| `median_unit_duration` | Median symbolic note duration. |
| `std_unit_duration` | Standard deviation of symbolic note durations. |
| `most_common_note` | Symbolic note occurring most frequently in the composition. |
| `most_common_note_count` | Frequency of the most common symbolic note. |

Together, these variables provide a compact statistical representation of each individual composition. They preserve both the musical metadata and the symbolic characteristics of the work, enabling subsequent corpus-level statistical analysis, visualization, clustering, and machine learning applications.

In [28]:
sample_file = txt_files[0]

sample_result = analyze_composition(sample_file)

sample_result_df = pd.DataFrame(
    sample_result.items(),
    columns=["Variable", "Value"],
)

sample_result_df.index = range(
    1,
    len(sample_result_df) + 1,
)

display(
    sample_result_df.style
    .hide(axis="index")
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ]
    )
)

Variable,Value
makam,acem
form,ilahi
usul,duyek
title,aldanma_dunya
composer,zekai_dede
filename,acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
note_count,271
unique_note_count,9
minimum_koma53,-1
maximum_koma53,358


## Preparing the Individual Analysis Output Directory

The statistical profile of each musical composition is stored separately to preserve the independence of the TXT files.

A dedicated output directory is therefore created under the processed data directory. Each generated CSV file will contain the statistical descriptors of one composition only. This organization prevents the individual musical works from being merged during the analysis stage and enables each result to be examined or reused independently.

In [29]:
individual_analysis_directory = (
    processed_data_directory / "individual_composition_analyses"
)

individual_analysis_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("Individual composition analysis directory initialized.")

Individual composition analysis directory initialized.


## Creating Safe Output Filenames

SymbTr composition filenames may contain spaces, accented characters, punctuation marks, or other symbols that are not fully compatible with filenames across different operating systems.

The function below converts each composition name into a standardized, ASCII-compatible filename by normalizing Unicode characters, removing unsupported symbols, replacing invalid characters with underscores, and eliminating redundant underscores. The resulting filename is used when saving the statistical profile of each composition as an individual CSV file named **`<composition_name>_analysis.csv`** in the **`data/processed/individual_composition_analyses`** directory.

This preprocessing step ensures that all generated output files have consistent, portable, and operating-system-independent filenames.

In [30]:
import re
import unicodedata


def create_safe_filename(file_stem):
    """
    Convert a composition filename into a safe output filename.
    """

    normalized_name = unicodedata.normalize(
        "NFKD",
        file_stem,
    )

    ascii_name = normalized_name.encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    safe_name = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        ascii_name,
    )

    safe_name = re.sub(
        r"_+",
        "_",
        safe_name,
    ).strip("_")

    return safe_name

## Analyzing Every Composition Independently

The validated analysis pipeline is now applied independently to every SymbTr TXT file contained in the dataset.

For each composition, the workflow performs the following operations:

- reads the symbolic music file;
- extracts the metadata encoded in the filename;
- computes descriptive statistics from the symbolic pitch (**Koma53**), note, and duration information;
- generates a statistical profile representing the individual musical composition;
- saves the resulting profile as an independent CSV file named **`<composition_name>_analysis.csv`** (e.g., `rast_sarki_analysis.csv`).

During processing, the notebook continuously reports the analysis progress by indicating how many compositions have been processed. The entire workflow is enclosed within an exception-handling mechanism to ensure that unreadable or irregular files do not interrupt the analysis of the remaining compositions. Files that cannot be processed are recorded separately together with the corresponding error information, enabling subsequent inspection and quality control.

The extracted statistical profiles are stored both in memory and as individual CSV files. Maintaining one analysis file per composition preserves a direct correspondence between each symbolic music file and its statistical representation. This organization improves traceability, facilitates reproducibility, and provides standardized feature representations that can subsequently be integrated for corpus-level statistical analysis, visualization, clustering, and machine learning applications.

**Output.** This cell analyzes every SymbTr TXT file independently, generates a statistical profile for each composition, and saves the results as individual CSV files named **`<composition_name>_analysis.csv`** in the **`individual_composition_analysis`** directory. Any files that cannot be processed are recorded together with the corresponding error information.

In [31]:
individual_results = []
failed_files = []

total_files = len(txt_files)

for index, file_path in enumerate(txt_files, start=1):

    try:
        result = analyze_composition(file_path)

        individual_results.append(result)

        result_table = pd.DataFrame(
            result.items(),
            columns=["Variable", "Value"],
        )

        safe_filename = create_safe_filename(file_path.stem)

        output_path = (
            individual_analysis_directory
            / f"{safe_filename}_analysis.csv"
        )

        result_table.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig",
        )

    except Exception as error:

        failed_files.append(
            {
                "filename": file_path.name,
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

    print(f"\rProcessed {index} of {total_files} compositions.", end="")

print("\nAnalysis completed.")
print(f"\rProcessed {total_files} of {total_files} compositions.")

print("\nAnalysis completed.\n")

print("=" * 60)
print("Individual Composition Analysis Summary")
print("=" * 60)
print(f"Total TXT files          : {total_files}")
print(f"Successfully analyzed    : {len(individual_results)}")
print(f"Failed analyses          : {len(failed_files)}")
print(f"Analysis files generated : {len(individual_results)}")
print(f"Output directory         : {individual_analysis_directory.name}")

Processed 3000 of 3000 compositions.
Analysis completed.
Processed 3000 of 3000 compositions.

Analysis completed.

Individual Composition Analysis Summary
Total TXT files          : 3000
Successfully analyzed    : 3000
Failed analyses          : 0
Analysis files generated : 3000
Output directory         : individual_composition_analyses


## Verifying the Generated Individual Analysis Files

After processing the complete SymbTr corpus, the notebook verifies that an individual statistical analysis file has been generated for every symbolic music composition.

The code below compares the number of original SymbTr TXT files with the number of generated **`*_analysis.csv`** files stored in the **`data/processed/individual_composition_analyses`** directory. Matching counts confirm that the analysis pipeline successfully produced one statistical profile for each composition.

This verification provides a final quality control step before proceeding to the corpus-level analyses presented in the following chapters.

In [32]:
generated_analysis_files = sorted(
    individual_analysis_directory.glob("*_analysis.csv")
)

print(f"Original TXT files          : {len(txt_files)}")
print(f"Generated analysis files    : {len(generated_analysis_files)}")

if len(generated_analysis_files) == len(txt_files):
    print("Verification successful: every composition has an analysis file.")
else:
    print("Warning: the number of generated files does not match the TXT files.")

Original TXT files          : 3000
Generated analysis files    : 3000
Verification successful: every composition has an analysis file.


## Inspecting a Generated Composition Analysis

The following table presents one of the statistical profiles generated during the analysis of the SymbTr corpus.

The profile combines the metadata extracted from the filename with the descriptive statistics computed from the symbolic note, **Koma53** pitch, and duration information. Displaying an example profile allows the generated output to be inspected before proceeding to corpus-level analyses.

The same statistical profile is also saved as an individual CSV file in the **`data/processed/individual_composition_analyses`** directory.

In [33]:
sample_result_df = pd.DataFrame(
    individual_results[0].items(),
    columns=["Statistic", "Value"],
)

sample_result_df.index = range(1, len(sample_result_df) + 1)

display(
    sample_result_df.style
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [("text-align", "left")],
            },
            {
                "selector": "td",
                "props": [("text-align", "left")],
            },
        ]
    )
)

,Statistic,Value
1,makam,acem
2,form,ilahi
3,usul,duyek
4,title,aldanma_dunya
5,composer,zekai_dede
6,filename,acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
7,note_count,271
8,unique_note_count,9
9,minimum_koma53,-1
10,maximum_koma53,358


## Creating the Corpus Summary

After generating the statistical profile for every SymbTr composition, the individual analysis results are combined into a single corpus summary table.

The `individual_results` object contains one statistical profile for each analyzed composition. These profiles are merged into a Pandas DataFrame, where each row represents a single symbolic music composition and each column corresponds to either metadata extracted from the filename or a statistical feature computed from the symbolic music data.

The combined dataset is saved as **`corpus_summary.csv`** in the **`data/processed`** directory:

```text
data/
└── processed/
    └── corpus_summary.csv
```

In addition, the statistical profile of every composition has already been saved as an independent CSV file in the following directory:

```text
data/
└── processed/
    └── individual_composition_analyses/
        ├── composition_1_analysis.csv
        ├── composition_2_analysis.csv
        ├── ...
        └── composition_3000_analysis.csv
```

Consequently, this notebook produces two complementary outputs:

- **Individual composition analysis files (`*_analysis.csv`)**, each containing the statistical profile of a single SymbTr composition.
- **`corpus_summary.csv`**, containing the statistical profiles of all analyzed compositions in a unified dataset.

The **`corpus_summary.csv`** dataset serves as the primary input for the corpus-level statistical analyses, visualizations, clustering methods, dimensionality reduction techniques, and machine learning applications presented in the subsequent chapters.

Finally, the notebook reports the dimensions of the generated corpus summary and displays the first **ten compositions** to verify that the summary table has been created successfully.

In [35]:
# Create the corpus summary table
corpus_summary = pd.DataFrame(individual_results)

# Define the output file
corpus_summary_path = (
    processed_data_directory
    / "corpus_summary.csv"
)

# Save the corpus summary
corpus_summary.to_csv(
    corpus_summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("Corpus summary successfully created.")
print(
    f"Corpus summary shape : "
    f"{corpus_summary.shape[0]} compositions × "
    f"{corpus_summary.shape[1]} variables"
)
print(f"Output file          : {corpus_summary_path.name}")

# Display the first ten compositions
preview = corpus_summary.head(10).copy()

preview.index = range(
    1,
    len(preview) + 1,
)

display(
    preview.style
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ]
    )
)

Corpus summary successfully created.
Corpus summary shape : 3000 compositions × 22 variables
Output file          : corpus_summary.csv


,makam,form,usul,title,composer,filename,note_count,unique_note_count,minimum_koma53,maximum_koma53,mean_koma53,median_koma53,std_koma53,koma53_range,total_unit_duration,minimum_unit_duration,maximum_unit_duration,mean_unit_duration,median_unit_duration,std_unit_duration,most_common_note,most_common_note_count
1,acem,ilahi,duyek,aldanma_dunya,zekai_dede,acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt,271,9,-1,358,317.527675,327.000000,60.452845,359,32.125000,0.031250,0.375000,0.118542,0.125000,0.072878,F5,59
2,acem,ilahi,nimevsat,calabim_bir,haci_bayram_veli,acem--ilahi--nimevsat--calabim_bir--haci_bayram_veli.txt,249,11,-1,367,323.686747,336.000000,48.714340,368,32.625000,0.031250,0.375000,0.131024,0.125000,0.090387,E5,66
3,acem,kupe,duyek,zulfunu,ahmet_avni_konuk,acem--kupe--duyek--zulfunu--ahmet_avni_konuk.txt,101,12,-1,362,308.376238,336.000000,86.064261,363,16.000000,0.062500,0.375000,0.158416,0.125000,0.072653,F5,20
4,acem,selam,devrikebir,asik-i_ger,huseyin_fahreddin_dede,acem--selam--devrikebir--asik-i_ger--huseyin_fahreddin_dede.txt,1161,21,-1,358,314.672696,327.000000,61.729343,359,169.125000,0.031250,0.750000,0.145672,0.125000,0.105376,C5,188
5,acem,seyir,sofyan,,sefik_gurmeric,acem--seyir--sofyan----sefik_gurmeric.txt,101,10,-1,358,325.514851,336.000000,48.468054,359,16.000000,0.062500,0.750000,0.158416,0.125000,0.118442,D5,21
6,acem,seyir,sofyan,1,erol_bingol,acem--seyir--sofyan--1--erol_bingol.txt,100,13,-1,371,328.240000,327.000000,37.182629,372,16.000000,0.062500,0.750000,0.160000,0.125000,0.108915,D5,18
7,acem,turku,duyek,ordunun_dereleri,ordu,acem--turku--duyek--ordunun_dereleri--ordu.txt,489,7,-1,349,310.893661,327.000000,73.306907,350,40.125000,0.031250,0.500000,0.082055,0.062500,0.061929,E5,136
8,acemasiran,agirsemai,senginsemai,ey_lebleri,dede_efendi,acemasiran--agirsemai--senginsemai--ey_lebleri--dede_efendi.txt,775,18,-1,362,314.397419,318.000000,45.651070,363,141.500000,0.062500,1.000000,0.182581,0.125000,0.133175,C5,104
9,acemasiran,aranagme,agiraksak,1,,acemasiran--aranagme--agiraksak--1--.txt,146,14,-1,380,324.616438,340.000000,70.764131,381,18.250000,0.062500,0.500000,0.125000,0.125000,0.072293,F5,25
10,acemasiran,aranagme,aksak,1,,acemasiran--aranagme--aksak--1--.txt,67,11,-1,358,309.835821,318.000000,58.143917,359,9.125000,0.125000,0.500000,0.136194,0.125000,0.056455,C5,10


## Description of the Corpus Summary Variables

The **`corpus_summary.csv`** file contains one row for each symbolic music composition analyzed in this notebook. Each column represents either metadata extracted from the filename or a statistical feature computed from the symbolic music data.

The table below serves as a data dictionary for the corpus summary by describing every variable included in the dataset. These variables are used throughout the subsequent chapters for corpus-level statistical analysis, visualization, clustering, dimensionality reduction, and machine learning applications.

Understanding the meaning of each variable facilitates the interpretation of the statistical analyses and supports the reproducibility of the computational workflow presented in this handbook.

In [36]:
variable_description = pd.DataFrame(
    {
        "Variable": [
            "makam",
            "form",
            "usul",
            "title",
            "composer",
            "filename",
            "note_count",
            "unique_note_count",
            "minimum_koma53",
            "maximum_koma53",
            "mean_koma53",
            "median_koma53",
            "std_koma53",
            "koma53_range",
            "total_unit_duration",
            "minimum_unit_duration",
            "maximum_unit_duration",
            "mean_unit_duration",
            "median_unit_duration",
            "std_unit_duration",
            "most_common_note",
            "most_common_note_count",
        ],
        "Description": [
            "Makam extracted from the filename.",
            "Musical form extracted from the filename.",
            "Usul extracted from the filename.",
            "Title of the composition extracted from the filename.",
            "Composer of the composition extracted from the filename.",
            "Original SymbTr TXT filename.",
            "Total number of symbolic notes in the composition.",
            "Number of distinct symbolic notes.",
            "Minimum Koma53 pitch value observed in the composition.",
            "Maximum Koma53 pitch value observed in the composition.",
            "Mean of the Koma53 pitch values.",
            "Median of the Koma53 pitch values.",
            "Standard deviation of the Koma53 pitch values.",
            "Difference between the maximum and minimum Koma53 pitch values.",
            "Sum of all symbolic unit durations.",
            "Minimum symbolic unit duration.",
            "Maximum symbolic unit duration.",
            "Mean symbolic unit duration.",
            "Median symbolic unit duration.",
            "Standard deviation of the symbolic unit durations.",
            "Most frequently occurring symbolic note.",
            "Number of occurrences of the most frequently occurring symbolic note.",
        ],
    }
)

variable_description.index = range(
    1,
    len(variable_description) + 1,
)

display(
    variable_description.style
    .set_properties(
        subset=["Variable"],
        **{
            "text-align": "left",
            "font-weight": "bold",
            "white-space": "nowrap",
        },
    )
    .set_properties(
        subset=["Description"],
        **{
            "text-align": "left",
            "white-space": "normal",
        },
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("width", "100%"),
                ],
            },
        ]
    )
)

,Variable,Description
1,makam,Makam extracted from the filename.
2,form,Musical form extracted from the filename.
3,usul,Usul extracted from the filename.
4,title,Title of the composition extracted from the filename.
5,composer,Composer of the composition extracted from the filename.
6,filename,Original SymbTr TXT filename.
7,note_count,Total number of symbolic notes in the composition.
8,unique_note_count,Number of distinct symbolic notes.
9,minimum_koma53,Minimum Koma53 pitch value observed in the composition.
10,maximum_koma53,Maximum Koma53 pitch value observed in the composition.


## Next Chapter

The next chapter, **Exploring and Statistically Analyzing the SymbTr TXT Collection**, presents a comprehensive statistical exploration of the complete SymbTr corpus. It examines the overall characteristics of the symbolic music collection through descriptive statistics, file distributions, musical attributes, and corpus-level properties. These analyses provide a quantitative understanding of the dataset and establish the foundation for the visualization, machine learning, and computational musicology studies presented in the subsequent chapters.